<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python & AI for Rates, Bonds, and Credit

## Chapter 4 · Yield Curves, Zero Rates, and Forward Rates

[Open in Google Colab](https://colab.research.google.com/github/yhilpisch/ratescode/blob/main/notebooks/ch04_yield_curves_zero_rates_forward_rates.ipynb)

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by different LLMs<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh

## Learning goals

A yield curve is the market's organized language for time, but the word covers several distinct objects: par curves, discount curves, zero curves, and forward curves. Chapter 4 treats them as consistent views of one another, converting discount factors into continuously compounded zero rates and interval forwards, and shows why interpolation in log-discount space is part of the model rather than a plotting choice. An official U.S. Treasury par-yield snapshot grounds the teaching curve in public data.

Afterwards you can convert any curve node among discount, zero, and forward representations, and explain why a forward rate is an implication of today's curve, not a forecast.

## Prepare the companion repository

This setup cell works with a local checkout or a fresh Google Colab runtime. In Colab it clones the public companion repository, then sets `ROOT` so the chapter code and frozen data snapshots are found without manual path edits.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

REPO_URL = "https://github.com/yhilpisch/ratescode.git"
candidates = [Path.cwd(), Path.cwd().parent,
              Path("/content/07_ratescode")]
ROOT = next((p.resolve() for p in candidates
             if (p / "code").is_dir()
             and (p / "data").is_dir()), None)
if ROOT is None:
    ROOT = Path("/content/07_ratescode")
    subprocess.run(["git", "clone", "--depth", "1",
                    REPO_URL, str(ROOT)], check=True)

packages = {"numpy": "numpy", "pandas": "pandas",
            "matplotlib": "matplotlib"}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install",
                    *missing], check=True)
print(f"Repository: {ROOT}")
print(f"Data files: {len(list((ROOT / 'data').glob('*.csv')))}")


## Run the chapter example

The chapter script loads the teaching zero-curve nodes in `ch04_zero_curve.csv` (maturities 0.5, 1, 2, and 5 years; continuously compounded zero rates from 3.5% to 4.2%), prints each node's discount factor and the continuously compounded forward over the following interval, and interpolates the 3-year discount factor in log space as `discount_3y`. Watch the forwards: they step up (0.039, 0.041, 0.044) even though the zero curve rises smoothly, and the script refuses maturities outside the node range rather than extrapolating.

In [ ]:
SCRIPT = ROOT / "code" / "ch04_yield_curves.py"
print(SCRIPT.relative_to(ROOT))

result = subprocess.run(
    [sys.executable, str(SCRIPT)],
    cwd=ROOT,
    capture_output=True,
    text=True,
    check=True,
)
print(result.stdout)

## Interpretation checklist

Verify the internal consistency of the printed curve before using it.

- Confirm each discount factor matches exp(-z*T): the 5-year node pairs the 0.042 zero with 0.810584, and factors decline monotonically from 0.982652.
- Check the interval forwards (0.039, 0.041, 0.044): with an upward-sloping zero curve, each forward exceeds the zero rate at the start of its interval.
- Read `discount_3y` 0.885148 as a log-discount interpolation between the 2- and 5-year nodes, and note that the script rejects maturities outside the node range instead of extrapolating silently.
- Keep the data provenance separate: the teaching nodes are illustrative, while the official Treasury par-yield CSV in `data/` carries a metadata sidecar and records its source and retrieval date.

In [ ]:
import pandas as pd

DATA = ROOT / "data"
chapter_files = sorted(DATA.glob("ch04_*.csv"))
summary = pd.DataFrame([
    {"file": path.name,
     "rows": len(frame := pd.read_csv(path)),
     "columns": len(frame.columns),
     "missing_cells": int(frame.isna().sum().sum()),
     "fields": str(list(frame.columns))}
    for path in chapter_files
])
summary


## Chapter-specific checks

Compare a synthetic zero curve with the latest official Treasury snapshot, then inspect the forward-rate implication of log-discount interpolation.

In [ ]:
import numpy as np

zero = pd.read_csv(DATA / "ch04_zero_curve.csv")
zero["discount"] = np.exp(-zero.zero_rate * zero.maturity)
zero["forward"] = -np.log(zero.discount).diff() / zero.maturity.diff()
official = pd.read_csv(DATA / "ch04_us_treasury_par_yield_curve_official.csv")
latest = official.tail(1).T.rename(columns={official.index[-1]: "latest"})
zero.round(6), latest.head(8)

## Mini-lab: Bootstrap two zero rates from par yields

The chapter treats the par curve as the observed market input and the discount curve as the internal pricing state. The mini-lab runs that conversion on the smallest possible case: two annual-pay par bonds with par yields of 3.7% (1-year) and 3.9% (2-year), each priced at 100 per 100 face. The 1-year par bond gives the first discount factor directly; the 2-year bond is then repriced to solve for the second. Both yields are illustrative inputs.

In [ ]:
import numpy as np

par_1y, par_2y = 0.037, 0.039  # annual par yields, decimals
d1 = 1.0 / (1.0 + par_1y)  # 1Y discount from the 1Y par bond
z1 = -np.log(d1)  # continuously compounded 1Y zero
coupon_2y = par_2y * 100.0  # 2Y par bond coupon cash
d2 = (100.0 - coupon_2y * d1) / (100.0 + coupon_2y)  # price at 100
z2 = -np.log(d2) / 2.0  # continuously compounded 2Y zero
print(f"1Y: discount {d1:.6f}, zero {z1:.6f}")
print(f"2Y: discount {d2:.6f}, zero {z2:.6f}")
z2 - z1  # zero-curve slope implied by the par yields

### What to observe

With these inputs the calculation yields discount factors 0.964320 and 0.926267, hence a continuously compounded 1-year zero of 0.036332 and a 2-year zero of 0.038296. Both zeros sit slightly below their par yields because an annually compounded rate converts to a smaller continuous one, and the 2-year zero exceeds the 1-year zero, as an upward-sloping par curve implies.

## Exercises

Use the mini-lab's par yields and the chapter script's curve table to complete these checks, stating the compounding basis with every rate.

1. **Steepen the curve.** Raise the 2-year par yield to 0.041 and recompute both zeros. *Hint:* Only `d2` and `z2` change; note the direction and size of the move.
2. **Compare bases.** Explain why the 1-year continuous zero 0.036332 differs from the 1-year par yield 0.037. *Hint:* `np.log(1.037)` converts an annually compounded rate to a continuous one.
3. **Add a forward rate.** From the two bootstrapped discount factors, compute the simple 1-year-into-1-year forward rate. *Hint:* `(d1 / d2 - 1) / 1` mirrors the forward logic in the chapter script.
4. **Interpolate like the script.** Reproduce the script's `discount_3y` of 0.885148 from the nodes in `ch04_zero_curve.csv`. *Hint:* Interpolate log discounts, not zero rates, using `np.interp` between the 2- and 5-year nodes.
5. **Reprice the par bonds.** Verify that discounting each bond's cash flows at your bootstrapped zeros reproduces prices of exactly 100. *Hint:* This is the bootstrap's built-in check: 3.9 * d1 + 103.9 * d2 must equal 100.

For more detail, see Chapter 4 of *Python & AI for Rates, Bonds, and Credit*.

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">